In [ ]:
import os, subprocess, sys
if not os.path.exists(os.path.join('src', 'ns5_core.py')):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Yukinoshita-lin/nsf5-steganography.git', '.'], check=True)
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
for mod, pkg in [('numpy', 'numpy'), ('PIL', 'Pillow'),
                 ('matplotlib', 'matplotlib'), ('sklearn', 'scikit-learn'),
                 ('joblib', 'joblib'), ('pandas', 'pandas')]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import numpy as np
print('project ready:', os.path.exists('src/ns5_core.py'))

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve
import numpy as np

X, y = make_classification(n_samples=600, n_features=8, n_informative=5,
                           n_redundant=2, random_state=0)
clf = LogisticRegression(max_iter=2000)
aucs = cross_val_score(clf, X, y, cv=5, scoring='roc_auc')
print('5-fold AUC: %.3f +/- %.3f' % (aucs.mean(), aucs.std()))

In [ ]:
# 演示：随机切分 vs 按组切分（虚构 photo_id）
rng = np.random.default_rng(0)
photo_id = np.repeat(np.arange(150), 4)          # 每张照片 4 个相似样本
X2, y2 = make_classification(n_samples=600, n_features=8, random_state=1)
# 让同一 photo 的样本互相复制特征 -> 同源泄漏时指标会虚高
X2 = X2.copy()
for pid in range(150):
    mask = photo_id == pid
    X2[mask] += rng.normal(0, 0.05, size=X2[mask].shape)  # 同源接近
from sklearn.model_selection import GroupKFold, KFold
print('random split AUC: %.3f' % np.mean(cross_val_score(clf, X2, y2, cv=KFold(5), scoring='roc_auc')))
print('grouped  AUC: %.3f' % np.mean(cross_val_score(clf, X2, y2, cv=GroupKFold(5), groups=photo_id, scoring='roc_auc')))

In [ ]:
proba = clf.fit(X, y).predict_proba(X)[:, 1]
fpr, tpr, th = roc_curve(y, proba)
j = tpr - fpr
best = th[np.argmax(j)]
print('AUC=%.3f Youden threshold=%.3f' % (roc_auc_score(y, proba), best))
print('随机切分指标虚高是典型的数据泄漏信号——真实实验必须按 photo_id 分组。')